In [ ]:
import kagglehub
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
# Download latest version
input_path = Path(
    kagglehub.dataset_download("mirichoi0218/insurance")
)
print("Path to dataset files:", input_path)

output_path = Path.cwd().parent / "outputs"
output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the dataset
df = pd.read_csv(input_path / "insurance.csv")

## 1. Exploratory Data Analysis

In [ ]:
# Display the first few rows and column information
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Set up the figure size
plt.figure(figsize=(15, 5))

# Subplot 1: Distribution of "charges"
plt.subplot(1, 3, 1)
sns.histplot(df["charges"], kde=True)
plt.title("Distribution des Frais Médicaux (charges)")
plt.xlabel("Frais Médicaux")
plt.ylabel("Fréquence")

# Subplot 2: "charges" vs "smoker" (Box plot)
plt.subplot(1, 3, 2)
sns.boxplot(x="smoker", y="charges", data=df)
plt.title("Charges vs Fumeur")
plt.xlabel("Fumeur (smoker)")
plt.ylabel("Frais Médicaux")

# Subplot 3: "charges" vs "age" (Scatter plot)
plt.subplot(1, 3, 3)
sns.scatterplot(x="age", y="charges", hue="smoker", data=df)
plt.title("Charges vs Âge (par Fumeur)")
plt.xlabel("Âge")
plt.ylabel("Frais Médicaux")

plt.tight_layout()
plt.savefig(output_path / "eda_insurance_1.png")

 
plt.figure(figsize=(12, 5))

# Subplot 1: "charges" vs "bmi" (Scatter plot, colored by smoker)
plt.subplot(1, 2, 1)
sns.scatterplot(x="bmi", y="charges", hue="smoker", data=df)
plt.title("Charges vs IMC (par Fumeur)")
plt.xlabel("IMC (bmi)")
plt.ylabel("Frais Médicaux")

# Subplot 2: "charges" vs "children" (Box plot)
sns.boxplot(x="children", y="charges", data=df)
plt.title("Charges vs Nombre d\"Enfants")
plt.xlabel("Nombre d\"Enfants (children)")
plt.ylabel("Frais Médicaux")

plt.tight_layout()
plt.savefig(output_path / "eda_insurance_2.png")


 


## 2. Modeling

In [ ]:
# Define features and target
X = df.drop("charges", axis=1)
y = df["charges"]

In [ ]:
# Apply log transformation to the target variable to handle skewness
# Add a small constant (1) to handle potential zero values, though not strictly necessary here
y_log = np.log1p(y)

In [ ]:
# Identify categorical and numerical columns
categorical_features = X.select_dtypes(include=["object"]).columns
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns

In [ ]:
# Create the preprocessing pipelines for numerical and categorical features
numerical_transformer = Pipeline(steps=[
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# --- Model Definition and Training ---
# Split the data
X_train, X_test, y_log_train, y_log_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
y_test = np.expm1(y_log_test) # Keep the original test target for final evaluation

# Model 1: Linear Regression
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

lr_pipeline.fit(X_train, y_log_train)
lr_pred_log = lr_pipeline.predict(X_test)
lr_pred = np.expm1(lr_pred_log) # Inverse transform predictions

# Model 2: Random Forest Regressor
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_log_train)
rf_pred_log = rf_pipeline.predict(X_test)
rf_pred = np.expm1(rf_pred_log) # Inverse transform predictions

# --- Evaluation ---

# Function to calculate and print metrics
def evaluate_model(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"\n--- {model_name} Evaluation ---")
    print(f"RMSE (Original Scale): ${rmse:,.2f}")
    print(f"R-squared (R²): {r2:,.4f}")
    return rmse, r2

lr_rmse, lr_r2 = evaluate_model(y_test, lr_pred, "Régression Linéaire")
rf_rmse, rf_r2 = evaluate_model(y_test, rf_pred, "Forêt Aléatoire")

# --- Feature Importance (from Random Forest) ---
# Get feature names after one-hot encoding
onehot_cols = lr_pipeline.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)
feature_names = list(numerical_features) + list(onehot_cols)

# Get importances from Random Forest
rf_importances = rf_pipeline.named_steps["regressor"].feature_importances_
importance_df = pd.DataFrame({"Feature": feature_names, "Importance": rf_importances})
importance_df = importance_df.sort_values(by="Importance", ascending=False)

# Display top 10 features
print("\n--- Feature Importance (Random Forest, Top 10) ---")
print(importance_df.head(10))
